# Chapter 7: Neural Networks

```{admonition} Learning Objectives
:class: tip
- Understand computational graphs and DAGs
- Master perceptron and multilayer perceptrons
- Apply activation functions (sigmoid, tanh, ReLU, softmax)
- Derive and implement backpropagation algorithm
- Use gradient descent variants (SGD, momentum, Adam)
- Understand forward and backward passes
- Implement vector-centric backpropagation
- Apply different loss functions
- Train neural networks effectively
```

```{epigraph}
When we talk mathematics, we may be discussing a secondary language built on the primary language of the nervous system.

-- John von Neumann
```

## 7.1 Introduction

Neural networks represent a natural generalization of the machine learning methods from Chapter 6. The key difference is that neural networks learn **complex nonlinear functions** by composing simpler functions in a layered architecture.

### From Linear Models to Neural Networks

**Linear Models** (Chapter 6):
$$y = f_W(\mathbf{x}) = \mathbf{W} \cdot \mathbf{x}$$

- Simple, interpretable
- Limited expressive power
- Cannot learn nonlinear patterns

**Neural Networks** (This chapter):
$$y = f_W(\mathbf{x}) = F(\mathbf{W}_k \cdots F(\mathbf{W}_2 F(\mathbf{W}_1 \mathbf{x})))$$

- Complex, nested composition
- High expressive power
- **Universal function approximators**

### Key Ideas

1. **Computational Graphs**: Represent complex functions as directed acyclic graphs (DAGs)
2. **Layers**: Organize computations hierarchically
3. **Nonlinear Activations**: Enable learning complex patterns
4. **Backpropagation**: Efficient algorithm for computing gradients
5. **Gradient Descent**: Optimize network parameters

### Universal Function Approximation

**Theorem** (Universal Approximation): A feedforward neural network with a single hidden layer containing a finite number of neurons can approximate any continuous function on a compact subset of $\mathbb{R}^n$, given appropriate activation functions.

**Implication**: Neural networks can learn arbitrarily complex patterns from data, given sufficient capacity and training data.

## 7.2 Computational Graphs

A **computational graph** is a directed acyclic graph (DAG) that represents a mathematical function through a composition of simpler operations.

### 7.2.1 Definition

**Definition** (Directed Acyclic Computational Graph): A computational graph contains:
- **Nodes**: Variables (input, hidden, output)
- **Edges**: Functional relationships with learnable parameters
- **Direction**: Forward flow from inputs to outputs
- **Acyclic**: No cycles (enables efficient gradient computation)

**Three Types of Nodes**:

1. **Input nodes**: Contain external inputs (no computation)
2. **Hidden nodes**: Compute intermediate values
3. **Output nodes**: Produce final outputs

**Node Computation**:

Each node $j$ computes: $y_j = f_j(\{y_i : (i,j) \in E\}, \{w_{ij} : (i,j) \in E\})$

where $f_j$ is the local function, $y_i$ are input node values, and $w_{ij}$ are edge weights.

### 7.2.2 Example Computational Graph

Consider the function:
$$f(x_1, x_2, x_3) = \ln(x_1 + x_2) + \exp(x_1 + x_2 + x_3) \cdot (x_2 + w_3 x_3)$$

**Decomposition**:
1. $h_1 = x_1 + x_2$
2. $h_2 = \ln(h_1)$
3. $h_3 = x_1 + x_2 + x_3$
4. $h_4 = \exp(h_3)$
5. $h_5 = x_2 + w_3 x_3$
6. $o = h_2 + h_4 \cdot h_5$

**Key Insight**: Complex function = composition of simple operations

### 7.2.3 Learning Parameters

Given training data $(\mathbf{x}^{(i)}, y^{(i)})$, learn weights $\mathbf{W}$ by minimizing loss:

$$\min_{\mathbf{W}} \sum_i \mathcal{L}(f_W(\mathbf{x}^{(i)}), y^{(i)})$$

where $\mathcal{L}$ measures mismatch between predictions and targets.

**Gradient Descent Update**:
$$\mathbf{W} \leftarrow \mathbf{W} - \eta \nabla_{\mathbf{W}} \mathcal{L}$$

**Challenge**: How to compute $\nabla_{\mathbf{W}} \mathcal{L}$ for complex graphs?

**Answer**: Backpropagation algorithm (Section 7.4)

## 7.3 Neural Networks as Computational Graphs

Neural networks are special cases of computational graphs with a **layered structure**.

### 7.3.1 Feedforward Neural Network

**Architecture**:
- Nodes organized in layers
- Connections only between adjacent layers
- Information flows forward: input → hidden → output

**Single Hidden Layer Network**:

$$\begin{align}
\mathbf{h} &= \sigma(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) \quad \text{(Hidden layer)} \\
\mathbf{o} &= \mathbf{W}_2 \mathbf{h} + \mathbf{b}_2 \quad \text{(Output layer)}
\end{align}$$

where:
- $\mathbf{x} \in \mathbb{R}^d$ is input
- $\mathbf{h} \in \mathbb{R}^p$ is hidden representation
- $\mathbf{o} \in \mathbb{R}^k$ is output
- $\sigma$ is activation function (applied element-wise)
- $\mathbf{W}_1 \in \mathbb{R}^{p \times d}$, $\mathbf{W}_2 \in \mathbb{R}^{k \times p}$ are weight matrices
- $\mathbf{b}_1, \mathbf{b}_2$ are bias vectors

### 7.3.2 Deep Neural Networks

**Multiple Hidden Layers**:

$$\begin{align}
\mathbf{h}^{(1)} &= \sigma(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) \\
\mathbf{h}^{(2)} &= \sigma(\mathbf{W}_2 \mathbf{h}^{(1)} + \mathbf{b}_2) \\
&\vdots \\
\mathbf{h}^{(k)} &= \sigma(\mathbf{W}_k \mathbf{h}^{(k-1)} + \mathbf{b}_k) \\
\mathbf{o} &= \mathbf{W}_{k+1} \mathbf{h}^{(k)} + \mathbf{b}_{k+1}
\end{align}$$

**Recursive Formulation**:
$$\mathbf{h}^{(\ell+1)} = \sigma(\mathbf{W}_{\ell+1} \mathbf{h}^{(\ell)} + \mathbf{b}_{\ell+1})$$

where $\mathbf{h}^{(0)} = \mathbf{x}$ (input layer).

### 7.3.3 Why Depth Matters

**Linear Networks Are Shallow**:

If $\sigma$ is identity (linear), then:
$$\mathbf{o} = \mathbf{W}_3 \mathbf{W}_2 \mathbf{W}_1 \mathbf{x} = \mathbf{W} \mathbf{x}$$

where $\mathbf{W} = \mathbf{W}_3 \mathbf{W}_2 \mathbf{W}_1$. This is just a single-layer network!

**Nonlinear Activations Enable Depth**:

With nonlinear $\sigma$, deeper networks can represent more complex functions efficiently.

**Example**: XOR problem
- Not linearly separable in input space
- Single hidden layer with 2 units and ReLU can solve it
- Linear network cannot solve it regardless of depth

## 7.4 Activation Functions

Activation functions introduce **nonlinearity** into neural networks, enabling them to learn complex patterns.

### 7.4.1 Common Activation Functions

**1. Sigmoid**:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Properties:
- Range: $(0, 1)$
- Smooth, differentiable
- **Gradient**: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$
- **Problem**: Vanishing gradients for large $|z|$

**2. Tanh** (Hyperbolic Tangent):
$$\tanh(z) = \frac{e^{2z} - 1}{e^{2z} + 1}$$

Properties:
- Range: $(-1, 1)$
- Zero-centered (better than sigmoid)
- **Gradient**: $\tanh'(z) = 1 - \tanh^2(z)$
- **Problem**: Still suffers from vanishing gradients

**3. ReLU** (Rectified Linear Unit):
$$\text{ReLU}(z) = \max(0, z)$$

Properties:
- Range: $[0, \infty)$
- **Gradient**: $\begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases}$
- Fast computation
- No vanishing gradient for positive values
- **Problem**: Dead neurons (always output 0)
- **Most popular** in deep learning

**4. Leaky ReLU**:
$$\text{LeakyReLU}(z) = \max(\alpha z, z)$$

where $\alpha \approx 0.01$.

Properties:
- Fixes dead neuron problem
- **Gradient**: $\begin{cases} 1 & \text{if } z > 0 \\ \alpha & \text{if } z \leq 0 \end{cases}$

**5. Hard Tanh**:
$$\text{HardTanh}(z) = \begin{cases} 1 & \text{if } z > 1 \\ z & \text{if } -1 \leq z \leq 1 \\ -1 & \text{if } z < -1 \end{cases}$$

Properties:
- Computationally cheaper than tanh
- Range: $[-1, 1]$

### 7.4.2 Softmax (Output Layer)

**Softmax** converts real values into probabilities for multi-class classification.

For $k$ classes with logits $z_1, ..., z_k$:

$$\sigma_i(\mathbf{z}) = \frac{e^{z_i}}{\sum_{j=1}^{k} e^{z_j}} \quad i = 1, ..., k$$

**Properties**:
- $\sum_{i=1}^k \sigma_i(\mathbf{z}) = 1$ (valid probability distribution)
- $\sigma_i(\mathbf{z}) \in (0, 1)$
- Differentiable
- Always used with **cross-entropy loss**

**Gradient** (special case with cross-entropy):
$$\frac{\partial \mathcal{L}}{\partial z_i} = \sigma_i - y_i$$

where $y_i$ is one-hot encoded target.

### 7.4.3 Activation Function Comparison

| Activation | Range | Gradient | Advantages | Disadvantages |
|------------|-------|----------|------------|---------------|
| Sigmoid | $(0,1)$ | $\sigma(1-\sigma)$ | Smooth | Vanishing gradient |
| Tanh | $(-1,1)$ | $1-\tanh^2$ | Zero-centered | Vanishing gradient |
| ReLU | $[0,\infty)$ | $\mathbb{1}_{z>0}$ | Fast, no vanishing | Dead neurons |
| Leaky ReLU | $(-\infty,\infty)$ | $\mathbb{1}_{z>0} + \alpha\mathbb{1}_{z\leq0}$ | Fixes dead neurons | Extra hyperparameter |
| Softmax | $(0,1)$ | Complex | Probabilities | Only for output layer |

**Rule of Thumb**:
- **Hidden layers**: ReLU or Leaky ReLU
- **Output layer (regression)**: Linear (identity)
- **Output layer (binary classification)**: Sigmoid
- **Output layer (multi-class)**: Softmax

## 7.5 Loss Functions

The loss function measures the mismatch between predictions and targets.

### 7.5.1 Regression Losses

**Mean Squared Error (MSE)**:
$$\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Gradient**:
$$\frac{\partial \mathcal{L}}{\partial \hat{y}_i} = -\frac{2}{n}(y_i - \hat{y}_i)$$

**Mean Absolute Error (MAE)**:
$$\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}}) = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**When to use**:
- MSE: Standard choice, penalizes large errors more
- MAE: More robust to outliers

### 7.5.2 Classification Losses

**Binary Cross-Entropy** (Logistic Loss):

For binary classification with $y \in \{0,1\}$ and prediction $\hat{y} = \sigma(z)$:

$$\mathcal{L}(y, \hat{y}) = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$$

Alternative formulation with $y \in \{-1, +1\}$:
$$\mathcal{L}(y, z) = \log(1 + e^{-yz})$$

**Categorical Cross-Entropy** (Multi-class):

For $k$ classes with one-hot encoded target $\mathbf{y}$ and softmax predictions $\hat{\mathbf{y}}$:

$$\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}}) = -\sum_{i=1}^{k} y_i \log(\hat{y}_i)$$

Since $\mathbf{y}$ is one-hot, if true class is $c$:
$$\mathcal{L} = -\log(\hat{y}_c)$$

**Gradient** (with softmax):
$$\frac{\partial \mathcal{L}}{\partial z_i} = \hat{y}_i - y_i$$

**Hinge Loss** (SVM):
$$\mathcal{L}(y, z) = \max(0, 1 - yz)$$

where $y \in \{-1, +1\}$ and $z$ is raw prediction.

### 7.5.3 Loss Function Selection

| Task | Output Activation | Loss Function |
|------|-------------------|---------------|
| Regression | Linear | MSE or MAE |
| Binary Classification | Sigmoid | Binary Cross-Entropy |
| Multi-class Classification | Softmax | Categorical Cross-Entropy |
| Multi-label Classification | Sigmoid (per class) | Binary Cross-Entropy (per class) |

## 7.6 Forward Propagation

**Forward propagation** computes the output of the network given an input.

### 7.6.1 Algorithm

```
Algorithm: FORWARD-PROPAGATION(Network, input x)

begin
    // Initialize
    h^(0) ← x
    
    // Compute each layer
    for ℓ = 1 to L do
        // Pre-activation (linear transformation)
        a^(ℓ) ← W_ℓ h^(ℓ-1) + b_ℓ
        
        // Post-activation (nonlinear transformation)
        h^(ℓ) ← σ_ℓ(a^(ℓ))
    
    // Output layer
    o ← h^(L)
    
    return o
end
```

### 7.6.2 Computational Details

**For each layer $\ell$**:

1. **Linear transformation**: $\mathbf{a}^{(\ell)} = \mathbf{W}_\ell \mathbf{h}^{(\ell-1)} + \mathbf{b}_\ell$
   - Matrix-vector multiplication: $O(p_{\ell-1} \cdot p_\ell)$
   - $p_\ell$ is number of units in layer $\ell$

2. **Nonlinear activation**: $\mathbf{h}^{(\ell)} = \sigma_\ell(\mathbf{a}^{(\ell)})$
   - Applied element-wise: $O(p_\ell)$

**Total Complexity**: $O(\sum_{\ell=1}^L p_{\ell-1} p_\ell)$

### 7.6.3 Example

**Network**:
- Input: $\mathbf{x} \in \mathbb{R}^3$
- Hidden: $\mathbf{h} \in \mathbb{R}^2$ with ReLU
- Output: $o \in \mathbb{R}$ with sigmoid

**Parameters**:
$$\mathbf{W}_1 = \begin{bmatrix} 2 & -1 & 0 \\ 1 & 3 & -2 \end{bmatrix}, \quad \mathbf{b}_1 = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$$
$$\mathbf{w}_2 = \begin{bmatrix} 1 \\ -1 \end{bmatrix}, \quad b_2 = 0.5$$

**Input**: $\mathbf{x} = [2, 1, 1]^T$

**Forward Pass**:

1. $\mathbf{a}^{(1)} = \mathbf{W}_1 \mathbf{x} + \mathbf{b}_1 = \begin{bmatrix} 2 & -1 & 0 \\ 1 & 3 & -2 \end{bmatrix} \begin{bmatrix} 2 \\ 1 \\ 1 \end{bmatrix} + \begin{bmatrix} 0 \\ 1 \end{bmatrix} = \begin{bmatrix} 3 \\ 4 \end{bmatrix}$

2. $\mathbf{h}^{(1)} = \text{ReLU}(\mathbf{a}^{(1)}) = \begin{bmatrix} 3 \\ 4 \end{bmatrix}$

3. $a^{(2)} = \mathbf{w}_2^T \mathbf{h}^{(1)} + b_2 = [1, -1] \begin{bmatrix} 3 \\ 4 \end{bmatrix} + 0.5 = -0.5$

4. $o = \sigma(a^{(2)}) = \frac{1}{1 + e^{0.5}} \approx 0.378$

**Store intermediate values** for backpropagation!

## 7.7 Backpropagation Algorithm

**Backpropagation** efficiently computes gradients of the loss with respect to all parameters using the **chain rule**.

### 7.7.1 The Chain Rule

For composed functions $y = f(g(x))$:
$$\frac{dy}{dx} = \frac{df}{dg} \cdot \frac{dg}{dx}$$

**Vector Case**: For $\mathbf{y} = f(\mathbf{h})$ and $\mathbf{h} = g(\mathbf{x})$:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = \left(\frac{\partial \mathbf{h}}{\partial \mathbf{x}}\right)^T \frac{\partial \mathcal{L}}{\partial \mathbf{h}}$$

where $\frac{\partial \mathbf{h}}{\partial \mathbf{x}}$ is the Jacobian matrix.

### 7.7.2 Backward Pass

**Key Idea**: Propagate gradients backward from output to input.

**Notation**:
- $\delta^{(\ell)} = \frac{\partial \mathcal{L}}{\partial \mathbf{a}^{(\ell)}}$ : gradient w.r.t. pre-activation
- $\mathbf{g}^{(\ell)} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}^{(\ell)}}$ : gradient w.r.t. post-activation

### 7.7.3 Backpropagation Equations

**Output Layer** ($\ell = L$):
$$\delta^{(L)} = \frac{\partial \mathcal{L}}{\partial \mathbf{o}}$$

Depends on loss function and output activation.

**Hidden Layers** ($\ell = L-1, ..., 1$):

1. **Post-activation to pre-activation**:
   $$\delta^{(\ell)} = \mathbf{g}^{(\ell)} \odot \sigma'(\mathbf{a}^{(\ell)})$$
   
   where $\odot$ is element-wise multiplication.

2. **Propagate to previous layer**:
   $$\mathbf{g}^{(\ell-1)} = \mathbf{W}_\ell^T \delta^{(\ell)}$$

**Weight Gradients**:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_\ell} = \delta^{(\ell)} (\mathbf{h}^{(\ell-1)})^T$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{b}_\ell} = \delta^{(\ell)}$$

### 7.7.4 Complete Backpropagation Algorithm

```
Algorithm: BACKPROPAGATION(Network, input x, target y)

begin
    // Forward pass (compute and store all activations)
    h^(0) ← x
    for ℓ = 1 to L do
        a^(ℓ) ← W_ℓ h^(ℓ-1) + b_ℓ
        h^(ℓ) ← σ_ℓ(a^(ℓ))
    o ← h^(L)
    
    // Compute loss
    L ← Loss(o, y)
    
    // Backward pass (compute gradients)
    // Initialize output gradient
    δ^(L) ← ∂L/∂o
    
    // Backpropagate through layers
    for ℓ = L down to 1 do
        // Gradient w.r.t. weights and biases
        ∂L/∂W_ℓ ← δ^(ℓ) (h^(ℓ-1))^T
        ∂L/∂b_ℓ ← δ^(ℓ)
        
        if ℓ > 1 then
            // Propagate to previous layer
            g^(ℓ-1) ← W_ℓ^T δ^(ℓ)
            δ^(ℓ-1) ← g^(ℓ-1) ⊙ σ'(a^(ℓ-1))
    
    return {∂L/∂W_ℓ, ∂L/∂b_ℓ for all ℓ}
end
```

**Complexity**: Same as forward pass, $O(\sum_{\ell=1}^L p_{\ell-1} p_\ell)$

## 7.8 Activation Function Derivatives

For backpropagation, we need derivatives of activation functions.

### 7.8.1 Derivative Formulas

**Sigmoid**:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
$$\sigma'(z) = \sigma(z)(1 - \sigma(z))$$

**Key property**: Derivative expressed in terms of output!

**Tanh**:
$$\tanh(z) = \frac{e^{2z} - 1}{e^{2z} + 1}$$
$$\tanh'(z) = 1 - \tanh^2(z)$$

**ReLU**:
$$\text{ReLU}(z) = \max(0, z)$$
$$\text{ReLU}'(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases} = \mathbb{1}_{z > 0}$$

**Leaky ReLU**:
$$\text{LeakyReLU}'(z) = \begin{cases} 1 & \text{if } z > 0 \\ \alpha & \text{if } z \leq 0 \end{cases}$$

### 7.8.2 Softmax with Cross-Entropy

**Special case**: When softmax output layer is paired with cross-entropy loss:

$$\frac{\partial \mathcal{L}}{\partial z_i} = \sigma_i(\mathbf{z}) - y_i$$

where $y_i$ is one-hot encoded target.

**Remarkably simple**! This is why softmax + cross-entropy is the standard for classification.

### 7.8.3 Backpropagation Table

| Layer Type | Forward | Backward (Gradient) |
|------------|---------|---------------------|
| Linear | $\mathbf{h} = \mathbf{W}\mathbf{x}$ | $\frac{\partial\mathcal{L}}{\partial\mathbf{x}} = \mathbf{W}^T \frac{\partial\mathcal{L}}{\partial\mathbf{h}}$ |
| Sigmoid | $h = \sigma(a)$ | $\frac{\partial\mathcal{L}}{\partial a} = \frac{\partial\mathcal{L}}{\partial h} \cdot h(1-h)$ |
| Tanh | $h = \tanh(a)$ | $\frac{\partial\mathcal{L}}{\partial a} = \frac{\partial\mathcal{L}}{\partial h} \cdot (1-h^2)$ |
| ReLU | $h = \max(0,a)$ | $\frac{\partial\mathcal{L}}{\partial a} = \frac{\partial\mathcal{L}}{\partial h} \cdot \mathbb{1}_{a>0}$ |
| Softmax + CE | $\sigma_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$ | $\frac{\partial\mathcal{L}}{\partial z_i} = \sigma_i - y_i$ |

## 7.9 Complete Training Example

Let's work through a complete forward and backward pass example.

### 7.9.1 Network Setup

**Architecture**:
- Input: $\mathbf{x} = [2, 1]^T$
- Hidden layer: 2 units with ReLU
- Output: 1 unit with sigmoid
- Target: $y = 1$
- Loss: Binary cross-entropy

**Parameters**:
$$\mathbf{W}_1 = \begin{bmatrix} 2 & -1 \\ 1 & 2 \end{bmatrix}, \mathbf{b}_1 = \begin{bmatrix} 0 \\ 1 \end{bmatrix}, \mathbf{w}_2 = \begin{bmatrix} 1 \\ -1 \end{bmatrix}, b_2 = 0$$

### 7.9.2 Forward Pass

**Layer 1 (Hidden)**:
$$\mathbf{a}^{(1)} = \mathbf{W}_1 \mathbf{x} + \mathbf{b}_1 = \begin{bmatrix} 2 & -1 \\ 1 & 2 \end{bmatrix} \begin{bmatrix} 2 \\ 1 \end{bmatrix} + \begin{bmatrix} 0 \\ 1 \end{bmatrix} = \begin{bmatrix} 3 \\ 5 \end{bmatrix}$$

$$\mathbf{h}^{(1)} = \text{ReLU}(\mathbf{a}^{(1)}) = \begin{bmatrix} 3 \\ 5 \end{bmatrix}$$

**Layer 2 (Output)**:
$$a^{(2)} = \mathbf{w}_2^T \mathbf{h}^{(1)} + b_2 = [1, -1] \begin{bmatrix} 3 \\ 5 \end{bmatrix} + 0 = -2$$

$$o = \sigma(a^{(2)}) = \frac{1}{1 + e^{2}} \approx 0.119$$

**Loss**:
$$\mathcal{L} = -[y \log(o) + (1-y)\log(1-o)] = -\log(0.119) \approx 2.13$$

### 7.9.3 Backward Pass

**Output Layer Gradient**:
$$\delta^{(2)} = \frac{\partial \mathcal{L}}{\partial a^{(2)}} = o - y = 0.119 - 1 = -0.881$$

**Weight Gradients (Layer 2)**:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}_2} = \delta^{(2)} \mathbf{h}^{(1)} = -0.881 \begin{bmatrix} 3 \\ 5 \end{bmatrix} = \begin{bmatrix} -2.643 \\ -4.405 \end{bmatrix}$$

$$\frac{\partial \mathcal{L}}{\partial b_2} = \delta^{(2)} = -0.881$$

**Backpropagate to Hidden Layer**:
$$\mathbf{g}^{(1)} = \mathbf{w}_2 \delta^{(2)} = \begin{bmatrix} 1 \\ -1 \end{bmatrix} (-0.881) = \begin{bmatrix} -0.881 \\ 0.881 \end{bmatrix}$$

**Apply ReLU Derivative**:
$$\delta^{(1)} = \mathbf{g}^{(1)} \odot \mathbb{1}_{\mathbf{a}^{(1)}>0} = \begin{bmatrix} -0.881 \\ 0.881 \end{bmatrix} \odot \begin{bmatrix} 1 \\ 1 \end{bmatrix} = \begin{bmatrix} -0.881 \\ 0.881 \end{bmatrix}$$

**Weight Gradients (Layer 1)**:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \delta^{(1)} \mathbf{x}^T = \begin{bmatrix} -0.881 \\ 0.881 \end{bmatrix} \begin{bmatrix} 2 & 1 \end{bmatrix} = \begin{bmatrix} -1.762 & -0.881 \\ 1.762 & 0.881 \end{bmatrix}$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{b}_1} = \delta^{(1)} = \begin{bmatrix} -0.881 \\ 0.881 \end{bmatrix}$$

### 7.9.4 Parameter Update

With learning rate $\eta = 0.1$:

$$\mathbf{W}_1 \leftarrow \mathbf{W}_1 - \eta \frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \begin{bmatrix} 2 & -1 \\ 1 & 2 \end{bmatrix} - 0.1 \begin{bmatrix} -1.762 & -0.881 \\ 1.762 & 0.881 \end{bmatrix} = \begin{bmatrix} 2.176 & -0.912 \\ 0.824 & 1.912 \end{bmatrix}$$

Similarly update $\mathbf{b}_1, \mathbf{w}_2, b_2$.

## 7.10 Gradient Descent Variants

### 7.10.1 Batch Gradient Descent

Compute gradient using **entire training set**:

$$\mathbf{W} \leftarrow \mathbf{W} - \eta \frac{1}{n} \sum_{i=1}^{n} \nabla_{\mathbf{W}} \mathcal{L}(f_W(\mathbf{x}^{(i)}), y^{(i)})$$

**Advantages**:
- Stable convergence
- Can use efficient matrix operations

**Disadvantages**:
- Slow for large datasets
- May get stuck in local minima
- Requires entire dataset in memory

### 7.10.2 Stochastic Gradient Descent (SGD)

Compute gradient using **single example**:

```
Algorithm: SGD(Training data D, learning rate η, epochs T)

begin
    Initialize W randomly
    
    for t = 1 to T do
        Shuffle D
        
        for each (x, y) in D do
            // Forward pass
            ỳ ← FORWARD-PROPAGATION(W, x)
            
            // Backward pass
            ∇_W L ← BACKPROPAGATION(W, x, y)
            
            // Update
            W ← W - η ∇_W L
    
    return W
end
```

**Advantages**:
- Fast updates
- Can escape local minima (due to noise)
- Online learning possible

**Disadvantages**:
- Noisy updates
- Slower convergence

### 7.10.3 Mini-Batch Gradient Descent

Compute gradient using **batch of $m$ examples** ($m \approx 32$-$256$):

$$\mathbf{W} \leftarrow \mathbf{W} - \eta \frac{1}{m} \sum_{i=1}^{m} \nabla_{\mathbf{W}} \mathcal{L}(f_W(\mathbf{x}^{(i)}), y^{(i)})$$

**Best of both worlds**:
- More stable than SGD
- Faster than batch GD
- Efficient with GPUs

**Standard choice** in deep learning!

### 7.10.4 Momentum

Add **momentum** term to smooth updates:

$$\begin{align}
\mathbf{v}_t &= \gamma \mathbf{v}_{t-1} + \eta \nabla_{\mathbf{W}} \mathcal{L} \\
\mathbf{W}_t &= \mathbf{W}_{t-1} - \mathbf{v}_t
\end{align}$$

where $\gamma \approx 0.9$ is momentum coefficient.

**Intuition**: "Rolling ball" that accumulates velocity

**Advantages**:
- Accelerates convergence
- Reduces oscillations
- Helps escape plateaus

### 7.10.5 Adam Optimizer

**Adam** (Adaptive Moment Estimation) combines momentum with adaptive learning rates:

```
Algorithm: ADAM(parameters W, learning rate α, β₁, β₂, ε)

begin
    m ← 0  // First moment (momentum)
    v ← 0  // Second moment (variance)
    t ← 0  // Time step
    
    while not converged do
        t ← t + 1
        g ← ∇_W L  // Compute gradient
        
        // Update biased first moment
        m ← β₁ m + (1 - β₁) g
        
        // Update biased second moment
        v ← β₂ v + (1 - β₂) g²
        
        // Bias correction
        m̂ ← m / (1 - β₁^t)
        v̂ ← v / (1 - β₂^t)
        
        // Update parameters
        W ← W - α m̂ / (√v̂ + ε)
    
    return W
end
```

**Default hyperparameters**:
- $\alpha = 0.001$ (learning rate)
- $\beta_1 = 0.9$ (momentum decay)
- $\beta_2 = 0.999$ (variance decay)
- $\epsilon = 10^{-8}$ (numerical stability)

**Most popular optimizer** in deep learning!

### 7.10.6 Optimizer Comparison

| Optimizer | Update Rule | Advantages | Disadvantages |
|-----------|-------------|------------|---------------|
| SGD | $\mathbf{W} - \eta \nabla \mathcal{L}$ | Simple, memory efficient | Slow, sensitive to $\eta$ |
| Momentum | $\mathbf{W} - \mathbf{v}$ | Faster, smoother | Extra hyperparameter |
| Adam | Adaptive per-parameter | Fast, robust | More memory, complex |

**Rule of thumb**: Start with Adam, fine-tune with SGD+momentum if needed.

## 7.11 Training Neural Networks

### 7.11.1 Complete Training Algorithm

```
Algorithm: TRAIN-NEURAL-NETWORK(Training data D, validation data V)

begin
    // Initialize
    W ← Initialize weights (Xavier or He initialization)
    best_val_loss ← ∞
    patience_counter ← 0
    
    for epoch = 1 to max_epochs do
        // Training phase
        Shuffle D
        for each mini-batch B in D do
            // Forward pass
            predictions ← FORWARD-PROPAGATION(W, B)
            loss ← COMPUTE-LOSS(predictions, targets)
            
            // Backward pass
            gradients ← BACKPROPAGATION(W, B)
            
            // Update weights
            W ← OPTIMIZER-UPDATE(W, gradients)
        
        // Validation phase
        val_loss ← EVALUATE(W, V)
        
        // Early stopping
        if val_loss < best_val_loss then
            best_val_loss ← val_loss
            best_W ← W
            patience_counter ← 0
        else
            patience_counter ← patience_counter + 1
            if patience_counter ≥ patience then
                break  // Stop training
    
    return best_W
end
```

### 7.11.2 Weight Initialization

**Problem**: Bad initialization can cause vanishing/exploding gradients.

**Xavier Initialization** (for tanh/sigmoid):
$$W_{ij} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n_{\text{in}} + n_{\text{out}}}}\right)$$

**He Initialization** (for ReLU):
$$W_{ij} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n_{\text{in}}}}\right)$$

where $n_{\text{in}}$ is number of input units.

**Bias**: Usually initialized to 0

### 7.11.3 Batch Normalization

Normalize activations in each layer:

$$\hat{\mathbf{h}} = \frac{\mathbf{h} - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

Then apply learnable scale $\gamma$ and shift $\beta$:
$$\mathbf{h}_{\text{BN}} = \gamma \hat{\mathbf{h}} + \beta$$

**Benefits**:
- Faster training
- Less sensitive to initialization
- Acts as regularization

### 7.11.4 Dropout

Randomly "drop" neurons during training:

```
During training:
    for each neuron i do
        with probability p:
            set h_i ← 0
        else:
            set h_i ← h_i / (1-p)

During testing:
    use all neurons (no dropout)
```

**Typical**: $p = 0.5$ for hidden layers

**Benefits**:
- Prevents overfitting
- Ensemble effect

### 7.11.5 Learning Rate Scheduling

**Step Decay**:
$$\eta_t = \eta_0 \cdot \gamma^{\lfloor t/k \rfloor}$$

**Exponential Decay**:
$$\eta_t = \eta_0 e^{-kt}$$

**Cosine Annealing**:
$$\eta_t = \eta_{\text{min}} + \frac{1}{2}(\eta_{\text{max}} - \eta_{\text{min}})\left(1 + \cos\left(\frac{t\pi}{T}\right)\right)$$

### 7.11.6 Hyperparameter Tuning

**Key hyperparameters**:
1. Learning rate $\eta$
2. Batch size $m$
3. Number of layers $L$
4. Layer sizes $p_1, ..., p_L$
5. Activation functions
6. Dropout rate $p$
7. Weight decay (L2 regularization)

**Tuning strategies**:
- Grid search
- Random search
- Bayesian optimization

## 7.12 Summary

### Key Concepts

1. **Computational Graphs**: DAGs representing function composition
2. **Feedforward Networks**: Layered architectures for learning complex functions
3. **Activation Functions**: Nonlinearity enables expressive power
4. **Backpropagation**: Efficient gradient computation via chain rule
5. **Gradient Descent**: Iterative optimization of parameters

### Algorithms

| Algorithm | Purpose | Complexity |
|-----------|---------|------------|
| Forward Propagation | Compute outputs | $O(\sum p_{\ell-1} p_\ell)$ |
| Backpropagation | Compute gradients | $O(\sum p_{\ell-1} p_\ell)$ |
| SGD | Update weights | $O(1)$ per weight |
| Adam | Adaptive updates | $O(1)$ per weight |

### Best Practices

1. **Architecture**:
   - Start simple, add complexity if needed
   - Use ReLU for hidden layers
   - Match output activation to task

2. **Training**:
   - Use Adam optimizer
   - Mini-batch size: 32-256
   - Monitor validation loss
   - Use early stopping

3. **Regularization**:
   - Dropout (0.5 for hidden layers)
   - Weight decay (L2: $10^{-4}$ to $10^{-5}$)
   - Batch normalization

4. **Debugging**:
   - Check gradients numerically
   - Visualize training curves
   - Start with small learning rate
   - Ensure loss decreases on training set

### When to Use Neural Networks

**Good for**:
- Large datasets
- Complex, nonlinear patterns
- Image, text, audio data
- When interpretability is not critical

**Not ideal for**:
- Small datasets (< 1000 examples)
- Need interpretability
- Limited computational resources
- Linear relationships

## 7.13 Implementation

For complete Python implementations, see:

[ch07_neural_networks_implementation.ipynb](ch07_neural_networks_implementation.ipynb)

The implementation notebook includes:

1. **Neural Network Framework**: Complete implementation from scratch
2. **Layer Types**: Dense, activation, softmax layers
3. **Activation Functions**: All functions with derivatives
4. **Loss Functions**: MSE, cross-entropy, hinge loss
5. **Optimizers**: SGD, momentum, Adam
6. **Backpropagation**: Numerical gradient checking
7. **Training Loop**: Complete with early stopping
8. **Applications**:
   - XOR problem (nonlinear classification)
   - MNIST digit classification
   - Regression on synthetic data
   - Iris classification with visualization

## Further Reading

### Textbooks

- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 7]
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
- Nielsen, M. (2015). *Neural Networks and Deep Learning*. [Free online]

### Classic Papers

- Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature*, 323(6088), 533-536.
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735-1780.
- He, K., et al. (2015). Deep residual learning for image recognition. *CVPR*.
- Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR*.

### Software Libraries

- **PyTorch**: [https://pytorch.org](https://pytorch.org)
- **TensorFlow**: [https://tensorflow.org](https://tensorflow.org)
- **JAX**: [https://github.com/google/jax](https://github.com/google/jax)
- **Keras**: [https://keras.io](https://keras.io)